# OCTA MAE — Phase 1 Pretraining

Trénuje ViT-Small encoder na jednotlivých OCTA snímkach (SVP + DCP ako samostatné vzorky).


## Imports + Cesty


In [3]:
import sys
from pathlib import Path

sys.path.insert(0, "scripts")
from octa_mae_core import *

ENCODER_ROOT = Path().resolve().parent 
RESULTS      = Path().resolve() / "results"  

EXCEL_PATH = ENCODER_ROOT / "data" / "master_excels" / "master_table.xlsx"
DATA_ROOT  = ENCODER_ROOT / "data"

print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM   : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
print(f"wandb  : {'available' if WANDB_AVAILABLE else 'not installed — logging disabled'}")
print(f"Excel  : {EXCEL_PATH}")
print(f"Data root: {DATA_ROOT}")

Device : cuda
GPU    : NVIDIA GeForce RTX 2050
VRAM   : 3.2 GB free / 4.0 GB total
wandb  : not installed — logging disabled
Excel  : C:\Users\klara\Desktop\Codes\03_Model_Training\data\master_excels\master_table.xlsx
Data root: C:\Users\klara\Desktop\Codes\03_Model_Training\data


---
## Phase 1 Baseline
Štandardný shuffle sampler. Berie všetky snímky kde `use_for_mae==1`.

In [4]:
p1_cfg = Phase1Config(
    excel_path  = str(EXCEL_PATH),
    data_root   = str(DATA_ROOT),
    output_base = str(RESULTS / "phase1" / "baseline"),

    # Tréning
    epochs        = 250,
    batch_size    = 64,
    lr            = 1.5e-4,
    warmup_epochs = 40,
    save_every    = 25,

    wandb_project  = "octa-mae",
    wandb_entity   = None,
    wandb_run_name = None,
)

print("Spúšťam Phase 1 Baseline...")
p1_encoder, p1_encoder_path = train_phase1(p1_cfg)

print(f"\n✅ Phase 1 Baseline hotovo!")
print(f"   Encoder uložený: {p1_encoder_path}")

2026-05-13 15:17:14,250 [INFO] ============================================================
2026-05-13 15:17:14,250 [INFO] PHASE 1 — BASELINE
2026-05-13 15:17:14,252 [INFO] ============================================================


Spúšťam Phase 1 Baseline...


2026-05-13 15:17:15,162 [INFO] Excel načítaný: 4199 riadkov | stĺpce: ['sample_id', 'dataset', 'patient_id', 'svp_path', 'dcp_path', 'has_dcp', 'label_raw', 'label_raw_dcp', 'label_clean_5class', 'height', 'width', 'split_mae', 'use_for_mae', 'split_full', 'use_for_cls_full', 'split_svp_dcp', 'use_for_cls_svp_dcp', 'split_svp_dcp_ballanced', 'use_for_cls_svp_dcp_ballanced']
2026-05-13 15:17:15,165 [INFO] MAE-eligible riadkov: 3753 (train: 3387, val: 366)
2026-05-13 15:17:15,223 [INFO] Phase1Dataset: 4695 vzoriek  (SVP=3350, DCP=1345)
2026-05-13 15:17:15,229 [INFO] Phase1Dataset: 503 vzoriek  (SVP=359, DCP=144)
2026-05-13 15:17:15,229 [INFO] Výstup: C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase1\baseline\20260513_151714
C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\scripts\octa_mae_core.py:937: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradS

KeyboardInterrupt: 

---
## Phase 1 Balanced
Každý batch obsahuje 50% SVP + 50% DCP vzorky. DCP sa oversampluje ak je ich menej.

In [6]:
# ── Config — zmeň čo potrebuješ ───────────────────────────────────────────────
p1b_cfg = Phase1BalancedConfig(
    excel_path  = EXCEL_PATH,
    data_root   = DATA_ROOT,
    output_base = str(RESULTS / "phase1" / "balanced"),

    # Tréning
    epochs        = 250,
    batch_size    = 64,
    lr            = 1.5e-4,
    warmup_epochs = 40,
    save_every    = 25,

    wandb_project  = "octa-mae",
    wandb_entity   = None,
    wandb_run_name = None,
)

print("Spúšťam Phase 1 Balanced...")
p1b_encoder, p1b_encoder_path = train_phase1_balanced(p1b_cfg)

print(f"\n✅ Phase 1 Balanced hotovo!")
print(f"   Encoder uložený: {p1b_encoder_path}")

2026-05-13 15:18:03,136 [INFO] ============================================================
2026-05-13 15:18:03,136 [INFO] PHASE 1 — BALANCED (50/50 SVP:DCP)
2026-05-13 15:18:03,136 [INFO] ============================================================


Spúšťam Phase 1 Balanced...


2026-05-13 15:18:03,652 [INFO] Excel načítaný: 4199 riadkov | stĺpce: ['sample_id', 'dataset', 'patient_id', 'svp_path', 'dcp_path', 'has_dcp', 'label_raw', 'label_raw_dcp', 'label_clean_5class', 'height', 'width', 'split_mae', 'use_for_mae', 'split_full', 'use_for_cls_full', 'split_svp_dcp', 'use_for_cls_svp_dcp', 'split_svp_dcp_ballanced', 'use_for_cls_svp_dcp_ballanced']
2026-05-13 15:18:03,655 [INFO] MAE-eligible riadkov: 3753 (train: 3387, val: 366)
2026-05-13 15:18:03,718 [INFO] Phase1BalancedDataset: 4695 vzoriek (SVP=3350, DCP=1345)
2026-05-13 15:18:03,723 [INFO] Phase1BalancedDataset: 503 vzoriek (SVP=359, DCP=144)
2026-05-13 15:18:03,725 [INFO] BalancedSVPDCPBatchSampler: 104 batchov | 32 SVP + 32 DCP na batch
2026-05-13 15:18:03,725 [INFO] Výstup: C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\results\phase1\balanced\20260513_151803
C:\Users\klara\Desktop\Codes\03_Model_Training\encoders\scripts\octa_mae_core.py:937: FutureWarning: `torch.cuda.amp.GradScaler(args...

KeyboardInterrupt: 